# Detecting AI-generated images with Defactify

This notebook trains and evaluates three models to distinguish real photographs from fully AI-generated images using the 96,000-image Defactify dataset.


## Setup

Set the dataset paths, random seed, training parameters, and execution switches. Select the available PyTorch device and configure offline loading when downloads are disabled.


In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (path / "README.md").exists() and (path / "PROJECT_CONTEXT.md").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the project root.")
DATA_DIR = PROJECT_ROOT / "data"
HF_HOME = DATA_DIR / "huggingface"
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_HOME / "datasets")
os.environ["TORCH_HOME"] = str(DATA_DIR / "torch")

DATASET_ID = "Rajarshi-Roy-research/Defactify_Image_Dataset"
DATASET_REVISION = "1c3b714c9691eb6dd11c2c1e92eab1ac29584fa0"
EXPECTED_SIZES = {"train": 42_000, "validation": 9_000, "test": 45_000}
SOURCE_NAMES = {0: "real", 1: "SD 2.1", 2: "SDXL", 3: "SD 3", 4: "DALL-E 3", 5: "Midjourney v6"}
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LEARNING_RATE = 1e-4
AUX_LOSS_WEIGHT = 0.2
RUN_LOAD_DATASET = True
RUN_DOWNLOAD = False
RUN_AUDIT = False
RUN_TRAIN = False
RUN_FINAL_TEST = False

# Set before importing datasets/huggingface_hub. Restart the kernel after changing mode.
if RUN_LOAD_DATASET and not RUN_DOWNLOAD:
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"
else:
    os.environ.pop("HF_HUB_OFFLINE", None)
    os.environ.pop("HF_DATASETS_OFFLINE", None)

import random
import numpy as np
import torch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"project={PROJECT_ROOT}; device={DEVICE}; torch={torch.__version__}")

project=/Users/dani/Documents/uni/4t/erasmus/applied/project; device=mps; torch=2.15.0


## Dataset

Load the pinned Defactify revision from the local cache or download it when enabled. Check the required columns and the published split sizes: 42,000 train, 9,000 validation, and 45,000 test images.


In [2]:
import shutil
from datasets import load_dataset, DownloadConfig

raw = None
if RUN_LOAD_DATASET:
    if RUN_DOWNLOAD:
        free_gib = shutil.disk_usage(PROJECT_ROOT).free / 2**30
        if free_gib < 20:
            raise RuntimeError(f"Only {free_gib:.1f} GiB free. Free at least 20 GiB before full download/cache preparation.")
    try:
        raw = load_dataset(
            DATASET_ID, name="default", revision=DATASET_REVISION,
            cache_dir=str(HF_HOME / "datasets"),
            download_config=DownloadConfig(local_files_only=not RUN_DOWNLOAD),
        )
    except (FileNotFoundError, OSError, ValueError) as exc:
        if not RUN_DOWNLOAD:
            raise RuntimeError(
                "The complete prepared dataset is not available in data/huggingface/. "
                "For the first download, restart the kernel and set RUN_DOWNLOAD=True."
            ) from exc
        raise
    sizes = {split: len(raw[split]) for split in raw}
    assert sizes == EXPECTED_SIZES, f"Unexpected published split sizes: {sizes}"
    assert sum(sizes.values()) == 96_000
    assert {"Caption", "Image", "Label_A", "Label_B"}.issubset(raw["train"].column_names)
    if not RUN_DOWNLOAD:
        cache_paths = [entry["filename"] for split in raw.values() for entry in split.cache_files]
        assert cache_paths and all(DATASET_REVISION in path for path in cache_paths), (
            "The local cache does not match the pinned dataset revision.")
    print("Loaded", sizes, "from", "download/cache" if RUN_DOWNLOAD else "local cache (offline)")
else:
    print("Dataset loading disabled. Set RUN_LOAD_DATASET=True to use the prepared local cache.")

/Users/dani/anaconda3/envs/applied/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 45000/45000 [00:06<00:00, 6478.50 examples/s] 


{'validation': 9000, 'train': 42000, 'test': 45000}


## Data checks

Check labels, image readability, and cross-split overlap in normalized captions and exact image hashes. Require the full dataset audit to pass before training.


In [ ]:
from collections import Counter, defaultdict
from hashlib import sha256
from io import BytesIO
from PIL import Image
import pandas as pd


def normalize_caption(value):
    return " ".join(str(value).casefold().split())


def image_bytes(field):
    # datasets.Image with decode=False returns {"bytes": ..., "path": ...}.
    if field["bytes"] is not None:
        return field["bytes"]
    return Path(field["path"]).read_bytes()


def audit_dataset(dataset):
    from datasets import Image as HFImage
    counts, formats, sizes = Counter(), Counter(), Counter()
    caption_splits, hash_splits = defaultdict(set), defaultdict(set)
    invalid = []
    for split in EXPECTED_SIZES:
        rows = dataset[split].cast_column("Image", HFImage(decode=False))
        for i, row in enumerate(rows):
            binary, source = int(row["Label_A"]), int(row["Label_B"])
            if binary not in (0, 1) or source not in SOURCE_NAMES or binary != int(source != 0):
                invalid.append((split, i, binary, source))
            counts[(split, binary, source)] += 1
            caption_splits[normalize_caption(row["Caption"])].add(split)
            blob = image_bytes(row["Image"])
            hash_splits[sha256(blob).hexdigest()].add(split)
            try:
                with Image.open(BytesIO(blob)) as img:
                    formats[(source, img.format)] += 1
                    sizes[(source, img.size)] += 1
                    img.verify()
            except Exception as exc:
                invalid.append((split, i, f"bad image: {exc}"))
    caption_overlap = sum(len(v) > 1 for v in caption_splits.values())
    image_overlap = sum(len(v) > 1 for v in hash_splits.values())
    summary = {
        "counts": pd.DataFrame([(s, b, src, n) for (s, b, src), n in sorted(counts.items())],
                               columns=["split", "binary", "source", "count"]),
        "formats_by_source": pd.DataFrame([(SOURCE_NAMES[src], fmt, n) for (src, fmt), n in sorted(formats.items())],
                                          columns=["source", "format", "count"]),
        "caption_overlap_groups": caption_overlap,
        "byte_identical_overlap_groups": image_overlap,
        "invalid_rows": invalid,
        "unique_dimensions_by_source": {SOURCE_NAMES[src]: len({size for (s, size) in sizes if s == src})
                                        for src in SOURCE_NAMES},
    }
    return summary


audit = None
if RUN_AUDIT:
    if raw is None:
        raise RuntimeError("Load the complete dataset first (RUN_DOWNLOAD=True).")
    audit = audit_dataset(raw)
    display(audit["counts"], audit["formats_by_source"])
    print({k: v for k, v in audit.items() if k not in ("counts", "formats_by_source")})
    assert not audit["invalid_rows"], "Invalid labels or unreadable images"
    assert audit["caption_overlap_groups"] == 0, "Caption-level cross-split leakage; investigate before training"
    assert audit["byte_identical_overlap_groups"] == 0, "Byte-identical cross-split images; investigate before training"
    assert sum(audit["counts"]["count"]) == 96_000
    source_totals = audit["counts"].groupby("source")["count"].sum().to_dict()
    assert source_totals == {source: 16_000 for source in SOURCE_NAMES}, source_totals
else:
    print("Full audit disabled; set RUN_AUDIT=True when ready to check the loaded dataset.")

## Image processing

Convert images to RGB tensors resized to 224 × 224, without augmentation. Build a DataLoader for each audited split, shuffling training data only.


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

image_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

class DefactifyTorchDataset(Dataset):
    def __init__(self, hf_split):
        self.rows = hf_split
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        return (image_transform(row["Image"].convert("RGB")),
                torch.tensor(int(row["Label_A"]), dtype=torch.float32),
                torch.tensor(int(row["Label_B"]), dtype=torch.long))

def make_loaders(dataset):
    return {split: DataLoader(DefactifyTorchDataset(dataset[split]),
                              batch_size=BATCH_SIZE, shuffle=(split == "train"),
                              num_workers=0, pin_memory=(DEVICE.type == "cuda"))
            for split in EXPECTED_SIZES}

loaders = make_loaders(raw) if raw is not None and audit is not None else None

## Models

Build the RGB ResNet-18 baseline and the RGB/Fourier models with concatenation or attention plus a six-class auxiliary source head. Check their output shapes with synthetic inputs without downloading pretrained weights.


In [ ]:
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights

RGB_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
RGB_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def normalize_rgb(x):
    return (x - RGB_MEAN.to(x.device)) / RGB_STD.to(x.device)

def fourier_map(x):
    gray = x.mean(dim=1, keepdim=True)
    spectrum = torch.fft.fftshift(torch.fft.fft2(gray, norm="ortho"), dim=(-2, -1))
    magnitude = torch.log1p(torch.abs(spectrum))
    return (magnitude - magnitude.mean(dim=(-2, -1), keepdim=True)) / magnitude.std(dim=(-2, -1), keepdim=True).clamp_min(1e-6)

class RGBBaseline(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.encoder = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.encoder.fc = nn.Linear(self.encoder.fc.in_features, 1)
    def forward(self, x):
        return self.encoder(normalize_rgb(x)).squeeze(-1), None

class TwoBranchDetector(nn.Module):
    def __init__(self, fusion="concat", auxiliary=False, pretrained=True):
        super().__init__()
        if fusion not in ("concat", "attention"):
            raise ValueError("fusion must be 'concat' or 'attention'")
        backbone = resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.rgb_encoder = nn.Sequential(*list(backbone.children())[:-2])
        self.frequency_encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((7, 7)),
        )
        self.fusion = fusion
        if fusion == "attention":
            self.rgb_projection = nn.Conv2d(512, 128, 1)
            self.cross_attention = nn.MultiheadAttention(128, num_heads=4, batch_first=True)
        self.head = nn.Linear(512 + 128, 1)
        self.source_head = nn.Linear(512 + 128, 6) if auxiliary else None
    def forward(self, x):
        rgb = self.rgb_encoder(normalize_rgb(x))
        freq = self.frequency_encoder(fourier_map(x))
        rgb_pooled = rgb.mean(dim=(-2, -1))
        if self.fusion == "attention":
            query = self.rgb_projection(rgb).flatten(2).transpose(1, 2)
            key_value = freq.flatten(2).transpose(1, 2)
            attended, _ = self.cross_attention(query, key_value, key_value, need_weights=False)
            freq_features = attended.mean(dim=1)
        else:
            freq_features = freq.mean(dim=(-2, -1))
        features = torch.cat([rgb_pooled, freq_features], dim=1)
        binary_logits = self.head(features).squeeze(-1)
        source_logits = self.source_head(features) if self.source_head is not None else None
        return binary_logits, source_logits

# Architecture smoke test needs no dataset and downloads no pretrained weights.
with torch.no_grad():
    sample = torch.rand(2, 3, IMAGE_SIZE, IMAGE_SIZE)
    for model in (RGBBaseline(pretrained=False), TwoBranchDetector("concat", pretrained=False),
                  TwoBranchDetector("attention", auxiliary=True, pretrained=False)):
        binary, source = model.eval()(sample)
        assert binary.shape == (2,)
        assert source is None or source.shape == (2, 6)
        print(type(model).__name__, getattr(model, "fusion", "rgb"), binary.shape,
              None if source is None else source.shape)

## Training and validation

Train the three models with a binary loss weighted using training class counts; the auxiliary model also predicts the source label. Record epoch metrics and keep the weights with the best validation balanced accuracy.


In [ ]:
from sklearn.metrics import balanced_accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix


def binary_metrics(labels, logits):
    labels = np.asarray(labels, dtype=int)
    logits = np.asarray(logits, dtype=float)
    probabilities = 1 / (1 + np.exp(-np.clip(logits, -80, 80)))
    predictions = (probabilities >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions,
                                                              average="binary", zero_division=0)
    return {"balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
            "precision_ai": float(precision), "recall_ai": float(recall), "f1_ai": float(f1),
            "auroc": float(roc_auc_score(labels, probabilities)) if len(np.unique(labels)) == 2 else None,
            "confusion_matrix_real_ai": confusion_matrix(labels, predictions, labels=[0, 1]).tolist()}


def run_epoch(model, loader, optimizer=None, pos_weight=None, auxiliary_weight=0.0):
    training = optimizer is not None
    model.train(training)
    all_labels, all_logits, total_loss = [], [], 0.0
    with torch.set_grad_enabled(training):
        for images, labels, sources in loader:
            images, labels, sources = images.to(DEVICE), labels.to(DEVICE), sources.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            binary_logits, source_logits = model(images)
            loss = nn.functional.binary_cross_entropy_with_logits(binary_logits, labels,
                                                                  pos_weight=pos_weight)
            if source_logits is not None and auxiliary_weight:
                loss = loss + auxiliary_weight * nn.functional.cross_entropy(source_logits, sources)
            if training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            all_labels.extend(labels.cpu().tolist())
            all_logits.extend(binary_logits.detach().cpu().tolist())
    return {"loss": total_loss / len(loader.dataset), **binary_metrics(all_labels, all_logits)}


def train_model(model, train_loader, validation_loader, epochs=EPOCHS):
    model = model.to(DEVICE)
    train_sources = np.asarray(train_loader.dataset.rows["Label_A"], dtype=int)
    real_count, ai_count = np.bincount(train_sources, minlength=2)
    pos_weight = torch.tensor([real_count / ai_count], device=DEVICE, dtype=torch.float32)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    best_score, best_state, history = -1.0, None, []
    for epoch in range(1, epochs + 1):
        train_result = run_epoch(model, train_loader, optimizer, pos_weight, AUX_LOSS_WEIGHT)
        validation_result = run_epoch(model, validation_loader, pos_weight=pos_weight,
                                      auxiliary_weight=AUX_LOSS_WEIGHT)
        history.append({"epoch": epoch, "train": train_result, "validation": validation_result})
        print(epoch, "train loss", round(train_result["loss"], 4),
              "validation balanced accuracy", round(validation_result["balanced_accuracy"], 4))
        if validation_result["balanced_accuracy"] > best_score:
            best_score = validation_result["balanced_accuracy"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history

experiments = {
    "rgb_baseline": lambda: RGBBaseline(pretrained=True),
    "rgb_fourier_concat": lambda: TwoBranchDetector("concat", pretrained=True),
    "rgb_fourier_attention_aux": lambda: TwoBranchDetector("attention", auxiliary=True, pretrained=True),
}
trained = {}
if RUN_TRAIN:
    if loaders is None:
        raise RuntimeError("Run the complete audit before training.")
    for name, factory in experiments.items():
        model, history = train_model(factory(), loaders["train"], loaders["validation"])
        trained[name] = {"model": model, "history": history}
else:
    print("Training disabled. Enable only after full audit passes.")

## Final evaluation

Select the model with the best validation balanced accuracy. When enabled, evaluate it on test and report binary metrics and recall by source.


In [ ]:
def evaluate_final(model, loader):
    model.eval()
    labels, sources, logits = [], [], []
    with torch.no_grad():
        for images, batch_labels, batch_sources in loader:
            batch_logits, _ = model(images.to(DEVICE))
            labels.extend(batch_labels.tolist())
            sources.extend(batch_sources.tolist())
            logits.extend(batch_logits.cpu().tolist())
    result = binary_metrics(labels, logits)
    predictions = (np.asarray(logits) >= 0).astype(int)
    result["recall_by_source"] = {
        SOURCE_NAMES[source]: float(np.mean(predictions[np.asarray(sources) == source] == int(source != 0)))
        for source in SOURCE_NAMES if np.any(np.asarray(sources) == source)
    }
    return result

if RUN_FINAL_TEST:
    if not trained or loaders is None:
        raise RuntimeError("Train and select a model using validation before final test.")
    chosen_name = max(trained, key=lambda name: max(
        step["validation"]["balanced_accuracy"] for step in trained[name]["history"]))
    final_result = evaluate_final(trained[chosen_name]["model"], loaders["test"])
    print("Selected on validation:", chosen_name)
    print(final_result)
else:
    print("Final test remains sealed.")

## References

- [Defactify dataset card](https://huggingface.co/datasets/Rajarshi-Roy-research/Defactify_Image_Dataset/blob/1c3b714c9691eb6dd11c2c1e92eab1ac29584fa0/README.md)
